


Building and Visualizing Data Sets
=============================


<div class="alert alert-block alert-info"> 
<h2>Overview</h2>

<strong>Questions:</strong>

* How can I use pandas to process data?

* How can I visualize relationships between different parts of my data?

* How can I integrate RDKit with pandas?

<strong>Objectives:</strong>

* Use pandas and seaborn to load and explore data

* Apply RDKit functions to build a data set in pandas

</div>

## Review of the Pandas data science library

[Pandas](https://pandas.pydata.org/docs/) is a Python library used for data analysis and manipulation. Within the world of data science, it is a ubiquitous and widely used library. If you are learning how to analyze data in Python, it will be almost impossible to avoid pandas. 

As with any python library, we need to import pandas to use its functions.  A common practice is to abbreviate `pandas` as `pd`. 

In [ ]:
import pandas as pd

The central data structure of pandas is called a DataFrame. Pandas DataFrames work very closely with NumPy arrays and Pandas dataframes are specifically for data which is two dimensional (rows and columns). NumPy arrays, while similar in some ways, can work with higher dimensional data.  Very broadly, NumPy arrays are designed to work with data that is all one data type, whereas Pandas dataframes are designed to work with a variety of data types.

Pandas is very powerful. In this session, we'll be learning how to access information in pandas dataframes and how to do some basic manipulation and analysis. We are going to be looking at a dataset which gives information about the elements in the periodic table.

In [ ]:
# Creating a pandas dataframe by reading in a CSV file.
df = pd.read_csv("data/PubChemElements_all.csv")

## Examining Data

Initially when loading data in, and also at certain points as we're working with it, we'll want to see what our dataframe looks like. You can see a preview of your dataframe using the `.head` function

When you preview the dataframe, you will see that the rows are labeled with numbers and the columns are labeled with column names.

In [ ]:
# The default behavior is to show the first five rows of a dataframe
# You can change this by putting a different number in the parenthesis
df.head()

The `.info` function will give information about the columns and the data type of those columns. The data type will become very important later as we work with data more.

In [ ]:
df.info()

For this dataframe, we see that the first column, `AtomicNumber` has the data type of `int64`. Here, `int` means `integer` and `64` means `64 bit`.  The `64 bit` refers to the amount of computer memory the variable can occupy. It won't really be important for us. Similarly, `float64` means `64 bit floating point`. These are decimal numbers.

The other column names which read `object` are not numeric. They might be strings or they might be something else. 

The `describe` function can be used on a dataframe to quickly see statistics about columns with numerical data. If you look at the columns that statistics are computed for and compare to the data type shown from `info`, you will see that we only get statistics for columns which had `int64` or `float64` data types.

In [ ]:
df.describe()

This information is extremely useful for understanding the data. We can also easily visualize the distribution of each column using Pandas's ``hist`` function.

In [ ]:
df.hist(figsize=(8,8), edgecolor='black', grid=False)

## Accessing Data

Pandas dataframes have names for rows (called the "index" in Pandas) and columns.

Pandas dataframes have rows and columns, you can see how many rows and columns using `.shape`. This will return the shape as `(num_rows, num_columns)`.

In [ ]:
df.shape

There are a few methods for accessing information in a Pandas dataframe, but the one that will be most important in this workshop is selecting particular columns of data.

In [ ]:
# Look at the atomic number column
# It won't show the whole column because it's too long
df["AtomicNumber"]

In [ ]:
# You could also show two columns at once
# You can combine with the .head() function to just show the beginning
df[["Symbol", "ElectronConfiguration"]].head() 

## Performing calculations with pandas: No more `for` loops!

Both pandas and NumPy dataframes have the convenient feature that they can do element-wise operations and use something called `broadcasting`. This means that if you are doing something like subtracting a number, multiplying, etc to a column or dataframe of information, it can be done all at once instead of with a `for` loop.  We saw this previously in our NumPy lesson.  

Consider if we wanted to calculate the melting point in degrees celsius for all of the elements. Instead of writing a `for` loop that does this, we can just write the following code. 

In [ ]:
df['MeltingPoint'] - 273.15

We could do this one two columns as well.  In that case, we would have a list of column names instead of just one column name.  This is why there are the double brackets in this notation; the inner brackets are specifying the list of column names. 

In [ ]:
df[['MeltingPoint', 'BoilingPoint']] - 273.15

We can save these in new dataframe columns

In [ ]:
df[["MeltingPointC", "BoilingPointC"]] = df[['MeltingPoint', 'BoilingPoint']] - 273.15

#Show the dataframe again to check our our new columns
df.head()

### The `.apply` method

The `.apply` method in pandas is used to apply a function along a row or column of a dataframe.
This is useful when you have a custom function that you need to use on every value in a column, but there is not a NumPy or Pandas function for it.

For example, we could apply the `len` function to our `Name` column to get the number of letters in the name for each element.

In [ ]:
# Number of letters in name - 
df["Name"].apply(len)

<div class="alert alert-block alert-success"> 
<strong>The .apply function</strong>

Notice that when we use `.apply`, we write the <strong>name</strong> of the function we want to apply, 
but we do not <strong>call</strong> the function. 
If we were to call the `len` function, we would use parentheses `()` with an argument.

</div>

### Applying a custom function

You can write your own function and use the apply function to use it. The following example classifies the melting point as above or below room temperature.

In [ ]:
def melting_point_category(temperature):
    if pd.isna(temperature):
        return "Unknown"
    elif temperature > 298.15:
        return "Above room temperature"
    else:
        return "At or below room temperature"

df["MeltingPointCategory"] = df["MeltingPoint"].apply(
    melting_point_category
)

df.head()

<div class="alert alert-block alert-warning">
<h3>Exercise</h3>

Write a custom function that accepts an element symbol and returns "One letter" or "Two letters." Apply it to `Symbol` to create `SymbolLengthCategory`. You will find the len function helpful.

</div>

In [ ]:
# Your code goes here

## Using RDKit Functions with Pandas DataFrames
Using `len` to calculate the number of letters in an atom name is a silly example, but for an example more related to our work with RDKit, let's add some additional atomic data.

RDKit has the ability to get information about atoms.
We can create a periodic table with `Chem.GetPeriodicTable`, then use associated functions to get information about atoms.

In [ ]:
from rdkit import Chem

In [ ]:
# Initialize the periodic table
periodic_table = Chem.GetPeriodicTable()

After we have a periodic table, we can apply functions from the periodic table to the atoms in our dataframe.

In [ ]:
df["NOuter"] = df["Symbol"].apply(periodic_table.GetNOuterElecs)
df.head()

<div class="alert alert-block alert-warning">
<h3>Exercise</h3>

Use the `tab` key on your periodic table object to check for other values you can calculate for atoms.
Pick one to add to your periodic table dataset.

</div>

In [ ]:
# Your code goes here

We can save a CSV file with our newly calculated values using the `to_csv` function.

In [ ]:
df.to_csv("data/periodic_data_processed.csv")

### Other calculations RDKit can perform

There are many other properties of molecules that RDKit can caculate.  In general, the methods in RDKit are organized into modules baesd on the type of property they calculate.  For instance, in some of the examples above, we used methods from the 
[`Descriptors` module](https://www.rdkit.org/docs/source/rdkit.Chem.Descriptors.html) and the [`rdMolDescriptors` module](https://www.rdkit.org/docs/source/rdkit.Chem.rdMolDescriptors.html).  You can click on either of those links to see the full list of the different properties you can access. 

Some methods you want to use exist on molecule objects themselves and there are other methods (e.g. in the Descriptors modules) where need to pass a molecule object to the method. 

For example if you have a molecule object called mol, you can use the mol.GetNumBonds() method to get the number of bonds. If you want to use RDKit's Chem.Descriptors.NumValenceElectrons, you have to pass it the mol object.

In [ ]:
mol = Chem.MolFromSmiles("CCO")
val_e = Chem.Descriptors.NumValenceElectrons(mol) # Can be used directly with the df.apply method, passing it a molecule object column. df["mol"].apply(Chem.Descriptors.NumValenceElectrons)
num_bonds = mol.GetNumBonds(onlyHeavy=False) # Needs to be wrapped in a function to be used with the apply method. df["mol"].apply(get_num_bonds) where get_num_bonds is a function that takes a molecule object and returns the number of bonds.

print(f"Number of valence electrons: {val_e}")
print(f"Number of bonds: {num_bonds}")

## Visualization

Visualizing data helps in understanding relationships and patterns that might not be apparent from raw data. Here, we will use Seaborn, a statistical visualization library, to create plots from our periodic table dataset. Seaborn is built on top of matplotlib, so if we would like to adjust any of the plots seaborn makes, we can do that through the Matplotlib interface we've used before.

We will start with a bar plot to show the ionization energy of elements across different group blocks:


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.catplot(data=df, x="NOuter", y="IonizationEnergy", kind="bar")
# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')

The plot above shows us periodic trends that we learned about in introductory chemistry. The two highest ionization energy categories correspond to elements with 2 valence electrons and 8 valence electrons, representing filled shells.

Seaborn can also allow us to easily create scatter plots to visualize relationships between continuous variables. For example, we can create a scatter plot to show the relationship between ionization energy and atomic radius:

In [ ]:
sns.scatterplot(data=df, x="AtomicRadius", y="Electronegativity", hue="GroupBlock")
plt.title('Electronegativity vs. Atomic Radius')
plt.xlabel('Atomic Radius')
plt.ylabel('Electronegativity')

<div class="alert alert-block alert-warning">
<h3>Exercise</h3>

Create a few other categorical plots to observe periodic trends.  Note, these instructions are given as y vs. x.

1. Electronegativity vs. Group Block as a bar plot.

2. Melting Point vs. Group Block as a bar plot.

3. Ionization Energy vs. Atomic Number as a scatter plot colored by GroupBlock.
   
</div>

Note that if you want to see some other types of plots that you can make in seaborn, you can find them at [seaborn plot types](https://seaborn.pydata.org/tutorial/function_overview.html).

In [ ]:
# Your code goes here.  Add additional cells if needed.

### Visualizing Correlation

A common way to visualize relationships between different categories of data categories is with a correlation plot.
The correlation matrix provides insights into the relationships between the variables. A correlation value close to 1 indicates a strong positive relationship, while a correlation value close to -1 indicates a strong negative relationship. A correlation value close to 0 indicates no relationship between the features.

In [ ]:
# Calculate the correlation matrix
corr = df.corr(numeric_only=True)
corr

Seaborn can be used to create a heatmap to allow easier examination of the correlation of different variables.

In [ ]:
# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")

The heatmap uses a "coolwarm" color scheme where red indicates positive correlation and blue indicates negative correlation between variables. Strongly correlated pairs are represented by darker shades of red, while strongly inversely correlated pairs are represented by darker shades of blue.

<div class="alert alert-block alert-warning">
<h3>Final Activity</h3>

In this activity we will create generate a dataset of amino acid data using RDKit and save it in a pandas dataframe.  There is a file called `amino_acids.txt` in the `data` folder that contains SMILES for the 20 naturally occurring amino acids. Your task is to create a pandas dataframe with the data listed below.  You should be able to use Pandas and the `.apply` function to add new columns of data to your DataFrame. 

The creation of a starting dataframe is done for you in the code cell below.  Add additional code to add the following columns to your dataframe:
1. The molecule object created from each SMILES string
2. Molecular Weight
3. Number of Heavy Atoms
4. One or two more molecular descriptors of your choosing

Make sure that you review the section called "### Other calculations RDKit can perform" above.

You will submit your code for this exercise.

</div>

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import PandasTools

# Ensure molecules are rendered in the notebook
PandasTools.RenderImagesInAllDataFrames(images=True)

filehandle = open("data/amino_acids.txt", "r")
amino_acid_data = filehandle.readlines()
filehandle.close()

# Delete the new line character at the end of each smiles string
smiles_list = []
for smiles in amino_acid_data:
    smiles = smiles.strip()
    smiles_list.append(smiles)

# Initialize the dataframe with the list
df = pd.DataFrame(smiles_list, columns=["SMILES"])
